In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
from segment_anything import sam_model_registry, SamPredictor

# --- 1. CONFIGURACIÓN DE RUTAS Y DISPOSITIVO ---
base_path = "/home/sergi/Escritorio/curso_ia"
checkpoint_path = os.path.join(base_path, "medsam_vit_b.pth")
image_path = os.path.join(base_path, "imagen_prueba.png")

# Detectar dispositivo (parche para evitar errores si no hay GPU NVIDIA configurada)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Ejecutando en: {device}")

# --- 2. CARGA DEL MODELO (CON PARCHE PARA CARGA EN CPU) ---
print("Cargando modelo MedSAM...")
medsam_model = sam_model_registry["vit_b"]()

try:
    with open(checkpoint_path, "rb") as f:
        # map_location="cpu" es vital para que no falle al buscar CUDA
        state_dict = torch.load(f, map_location="cpu")
    medsam_model.load_state_dict(state_dict)
    medsam_model.to(device)
    medsam_model.eval()
    print("¡Modelo MedSAM cargado con éxito!")
except Exception as e:
    print(f"Error al cargar el modelo: {e}")

# --- 3. PREPARACIÓN DE LA IMAGEN ---
try:
    image_pil = Image.open(image_path).convert("RGB")
    image_np = np.array(image_pil)
    
    predictor = SamPredictor(medsam_model)
    predictor.set_image(image_np)
    print("Imagen preparada para análisis de posicionamiento.")
except Exception as e:
    print(f"Error con la imagen: {e}")

# --- 4. DEFINICIÓN DEL PROMPT Y LÓGICA DE VALIDACIÓN ---
# [x_min, y_min, x_max, y_max] - Ajustado al dedo medio de tu imagen
input_box = np.array([280, 120, 480, 350]) 

print("Analizando anatomía...")
masks, scores, _ = predictor.predict(
    point_coords=None,
    point_labels=None,
    box=input_box[None, :],
    multimask_output=False,
)

score = scores[0]

# Definimos si la posición es apta basado en el reconocimiento del modelo
# Umbral sugerido: 0.6 (ajustable según iluminación)
es_valido = score > 0.6
estado_txt = "POSICIÓN VÁLIDA" if es_valido else "REVISAR POSICIÓN / ILUMINACIÓN"
color_display = 'green' if es_valido else 'red'

# --- 5. VISUALIZACIÓN ORIENTADA A ASISTENTE DE RAYOS X ---
plt.figure(figsize=(10, 10))
plt.imshow(image_np)

# Dibujar la máscara con color condicional
mask = masks[0]
h, w = mask.shape[-2:]
# Verde si reconoce bien la anatomía, Rojo si la confianza es baja
rgba_color = np.array([0, 1, 0, 0.4]) if es_valido else np.array([1, 0, 0, 0.4])
mask_image = mask.reshape(h, w, 1) * rgba_color.reshape(1, 1, -1)
plt.gca().imshow(mask_image)

# Dibujar el Bounding Box (simulando el área de colimación del equipo de rayos)
x0, y0, x1, y1 = input_box
plt.gca().add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0, 
                                  edgecolor='yellow', facecolor='none', lw=3, linestyle='--'))

# Título informativo para el informe
plt.title(f"Asistente de Posicionamiento Pre-Radiográfico\n{estado_txt} (Score: {score:.4f})", 
          fontsize=14, color=color_display, fontweight='bold')

plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Análisis finalizado. Resultado: {estado_txt}")